## DataFrames 101 — Creating and Exploring Data




## How to build DataFrames from CSV, JSON, and Parquet, and how to inspect schemas
Yesterday we explored RDDs — the low-level roots of Spark. They’re powerful, but a bit raw. Today, we step into the world you’ll spend most of your Spark life in: DataFrames.

Think of DataFrames as the “polished” version of RDDs. They look and feel like pandas DataFrames but scale like Spark. They carry not just data but also schema information (column names and types). That schema unlocks Spark’s optimizer, letting it figure out the best way to run your queries.

By the end of this session, you’ll know how to create DataFrames from multiple file formats and explore them like a pro.

### What is a DataFrame?
A DataFrame is a distributed collection of rows with a known schema. Each row is like a Python dictionary or a database record.

If an RDD is just a bag of objects, a DataFrame is a table with structure:

- Columns have names and data types.
- Spark can reason about queries because it knows the schema.
- You can use both an API (like .select, .filter) and SQL on the same data.

### Creating Your First DataFrame
The entry point is always your SparkSession.

In [0]:

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("day6-demo") \
    .getOrCreate()

# Create DataFrame from Python list
data = [("North", 1000), ("South", 1500), ("East", 1200)]
columns = ["region", "revenue"]

df = spark.createDataFrame(data, columns)
df.show()

### Congratulations — that’s your first Spark DataFrame!

Reading from Files
Most real-world DataFrames won’t come from lists in your notebook. They’ll come from files sitting in data lakes, HDFS, or cloud storage. Spark supports many formats out of the box. Let’s go through the common ones.

### 1. Reading CSV

In [0]:
df_csv = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("/Volumes/thedataengineering_00/data/sampledata/sampleimages/clean_sales.csv")

df_csv.show(5)

.option("header", True) tells Spark to use the first row as column names.

.option("inferSchema", True) tries to guess column types. (Handy for learning, but in production, you’ll often define schemas explicitly for reliability.)

### 2. Reading JSON

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

sales_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True),
    StructField("city", StringType(), True),
    StructField("payment_method", StringType(), True)
])

In [0]:
json_path = "/Volumes/thedataengineering_00/data/sampledata/sampleimages/sales.json"

sales_df = (
    spark.read
         .schema(sales_schema)
         .option("mode", "PERMISSIVE")   # keeps malformed rows
         .json(json_path)
)

display(sales_df)

JSON is common in event logs or API data. Spark can parse nested structures too — we’ll cover that later when we talk about complex types.

### 3. Reading Parquet

In [0]:
df = spark.read.csv("/Volumes/thedataengineering_00/data/sampledata/sampleimages/sales.json", header=True, inferSchema=True)
df.write.format("parquet").mode("overwrite").save("/Volumes/thedataengineering_00/data/sampledata/sampleimages/sale_parquet.parquet")


In [0]:
df_parquet = spark.read.parquet("/Volumes/thedataengineering_00/data/sampledata/sampleimages/sale_parquet.parquet")
df_parquet.show(5)

Parquet is a columnar format — it’s compact, efficient, and supports schema evolution. This is the format you’ll see most often in production pipelines.

### Exploring a DataFrame

Once you have a DataFrame, the first thing you’ll want to do is peek inside. Spark gives you several tools for that.

In [0]:
# Show rows
df_csv.show(5)

# Print schema
df_csv.printSchema()

# See column names
print(df_csv.columns)

# Count rows (this triggers computation)
print("Number of rows:", df_csv.count())

# Describe numeric columns
df_csv.describe().show()

### Quick Transformations While Exploring

Exploring often means trying out small transformations

In [0]:
# Select specific columns
df_csv.select("name", "amount").show(5)

# Filter rows
df_csv.filter(df_csv["amount"] > 12000).show()

# Rename columns
df_csv = df_csv.withColumnRenamed("revenue", "total_revenue")

### Mini Example: CSV → Clean → Explore

Let’s put it all together in a small example. Imagine we have a sales.csv like this:

In [0]:
# Step 1: Load
df = spark.read.option("header", True).option("inferSchema", True).csv("/Volumes/thedataengineering_00/data/sampledata/sampleimages/clean_sales.csv")

# Step 2: Explore schema
df.printSchema()

# Step 3: Clean missing values
df = df.na.fill({"amount": 0})

# Step 4: Peek at data
df.show()

# Step 5: Simple aggregation
df.groupBy("name").sum("amount").show()

### Wrapping Up

Today you learned the foundations of DataFrames:

- They’re structured, schema-aware datasets — like big, distributed tables.
- You can create them from Python objects or read from CSV, JSON, Parquet, and more.
- Spark gives you handy tools to explore: .show(), .printSchema(), .describe().
- Every time you transform a DataFrame, Spark builds a plan — execution happens only when you call an action.

If RDDs were the raw clay, DataFrames are the shaped bricks. Tomorrow, we’ll learn how to actually build with those bricks — applying transformations to shape your data into useful forms.

That’s Day 6. You now know how to create and explore Spark DataFrames.

Next up: Day 7 — Transformations in Spark: select, filter, withColumn. We’ll finally start sculpting real data.